# 01 · 數據:怎麼選、怎麼拼
**這本講我們怎麼挑數據、怎麼把多源拼成一張表。** 不寫代碼也能看懂;想自己從原始數據集建緩存,
需要下載「上海城市數據集」(見 `數據集說明.md`)。沒下也沒關係——隨包已帶 3 站緩存,後面幾本都能離線跑。

> 爲什麼單獨講數據?因爲「權力 → 形態」的可信度,**全押在數據怎麼選**。
> OSM 版(新加坡)tag 太稀疏,36% 的樓只能標 unknown;上海這套換成 5 個各司其職的本地來源,unknown 壓到 1–6%。

## 怎麼執行
- 點一格,按 **Shift+Enter** 執行;或選單 Run → Run All Cells 從頭跑。
- 結果與圖會顯示在該格下面。代碼都在 `engine/` 文件夾,平時不用打開。

In [ ]:
# 讓 notebook 找到 engine 裏的代碼(這格不用改,直接執行)
import sys, os
sys.path.insert(0, os.path.abspath("engine"))
# 清掉舊的引擎模塊緩存:改了 config/engine 後,重跑本格即刻生效(免重啟內核)
for _m in [k for k in list(sys.modules) if k in ("config", "common", "operators", "measure")
           or k == "plots" or k.startswith("plots.") or k == "prints" or k.startswith("prints.")]:
    del sys.modules[_m]
import config, common as C, plots, prints
prints.ready()
plots.capture("01")   # 開啟自動存圖:本冊每張圖都會存到 out/<slug>/Step_01/<時間戳>/

## 一、爲什麼要"多源"
一個來源講不清權力。我們要回答三件事,各需不同數據:
1. **樓長什麼樣、多高?** —— 幾何 + 實測高度。
2. **這塊地是幹嘛的(政府?商業?居住?)** —— 離散的用途主鍵。
3. **政府辦公 vs 商務辦公怎麼分開?** —— 用途類別要**自帶政商之分**(OSM 的 `office=*` 做不到)。

所以用**級聯**:一個主鍵來源打底,覆蓋不到的地方再用次級來源補。

## 二、5 個 spine 文件(每個爲什麼選它)
| 角色 | 文件 | 給什麼 | 爲什麼選它 |
|---|---|---|---|
| **幾何+高度** | `01-建築輪廓/②AI解析/…-帶高度` | footprint + **實測** Height(0–338m) | 終於有真超高層 token(陸家嘴 338m),不再靠樓層×3.5 估 |
| **離散主鍵** | `09-開源土地利用/…開源建設用地`(EULUC) | 地塊級用途 class2 | 面狀、覆蓋 90%+;`行政辦公`501 vs `商務辦公`201 **類別裏就分政商** |
| **補充用途** | `01-建築輪廓/③其它/…-帶年份`(Function) | 建築級 Function/Age | EULUC 沒覆蓋到處補;Office 不分政商,故只作補充 |
| **輔助** | `02-POI&AOI/…/百度AOI` | type/結構/價格/時間 | 再補一層;價格/結構僅作離散 tag、不外發原值 |
| **範圍** | `03-行政區/…/鄉鎮街道邊界` | 街道多邊形 | 研究單位 = 街道(非方形、變大小),不是任意方框 |

In [ ]:
# 看 5 個 spine 文件的路徑,以及你機器上是否有(沒有=還沒下數據集,用隨包緩存即可)
prints.spine_paths()

## 三、多源怎麼"拼"在一起(空間 join)
樓是**點/面**,用途是**地塊面**。拼法:取每棟樓的代表點(質心),看它**落在**哪塊用途面裏
(`within`),把那塊的用途貼回這棟樓。AOI 同理。這叫**空間連接 spatial join**。

然後**級聯取第一個命中**:先問 EULUC,有就定;沒有再問 Function;再沒有問 AOI;都沒有 = unknown。
一棟樓最終只拿到**一個**用途來源的判定——保持離散、可讀、可爭論。

## 四、親手建緩存(需要數據集;沒下就跳過這格)
把 `config.py` 的 `DATASET_ROOT` 指到你解壓的「上海城市數據集」根目錄,存檔、回到第二格重跑,再跑這格:
它會對當前 `SLUG` 做**裁切 + 多源 join**,寫出 `data/<slug>/buildings.parquet`。
沒設 `DATASET_ROOT` 時,這格會自動**改用隨包緩存**,不報錯。

In [ ]:
# 有數據集 → 親手建緩存;沒有 → 用隨包緩存。兩條路都通(分支細節在 C.build_or_load 裏)
df, source = C.build_or_load(config.SLUG)
prints.prepared(df, source)

## 五、看一眼拼出來的表
每棟樓現在帶着多源信息:`height_m`(實測高度)、`euluc`(地塊用途)、`function`、`aoi_*`。
後面「怎麼 mapping」那本,就靠這幾列把每棟樓對應到一個權利方。

In [ ]:
# 看一眼拼出來的表:多源覆蓋率 + 前 8 棟,再畫一張 footprint + 實測高度分布
prints.coverage(df)
plots.data_overview(df)

## 誠實邊界(數據這一關)
- **EULUC 是地塊(面)級、優先於建築級 Function**:居住地塊裏零星的小公建會被併入「居民」。
- **danwei 看不見**:工人新村是國家/單位建的,但用途=居住 → 會被算成居民;只有形態記得它是單位建的。
- **informal 本數據無信號 → 恆爲空**,不從形態硬猜。
- AI 高度對極端超高層(>~340m)可能低估;AOI 價格/結構僅作離散 tag、不外發原值。

> 這些不是 bug,是**把簡化講清楚**。下一本:[`02 映射`] —— 這張多源表怎麼變成「誰算誰的」。